In [ ]:
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
!chmod +x cloudflared

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 39.3M  100 39.3M    0     0  27.0M      0  0:00:01  0:00:01 --:--:--  109M


In [ ]:
# ----------------------------------------------------------------------
# Cloudflare Tunnel 버전 OCR 서버
# ----------------------------------------------------------------------

from flask import Flask, request, Response
from flask_cors import CORS
from PIL import Image
import numpy as np
import cv2
import torch
import gc

# Qwen2-VL
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from qwen_vl_utils import process_vision_info


# ======================================================================
# 1. 모델 로드
# ======================================================================
print("모델 로딩 중...")
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()
print("✅ 모델 로드 완료!")


# ======================================================================
# 2. OCR 함수
# ======================================================================
def run_qwen_ocr(image):
    prompt = """
    거래 일시: [YYYY-MM-DD HH:MM:SS]
    상호명: [가게이름]
    총액: [숫자만]
    기타 정보: [모든 인식 텍스트]
    """

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)

    inputs = processor(text=[text_input], images=image_inputs, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output = model.generate(**inputs, max_new_tokens=256)
    output_text = processor.decode(output[0], skip_special_tokens=True)

    return output_text


# ======================================================================
# 3. Flask API
# ======================================================================
app = Flask(__name__)
CORS(app)

@app.route("/predict", methods=["POST"])
def predict():
    file = request.files.get("file")
    if not file:
        return "파일 없음", 400

    img = Image.open(file.stream).convert("RGB")

    result = run_qwen_ocr(img)
    return Response(result, mimetype="text/plain")


# ======================================================================
# 4. Cloudflare Tunnel 실행
# ======================================================================
import subprocess
import threading
import time
import requests

def run_cloudflare():
    tunnel = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:5000", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

def find_url():
    time.sleep(3)
    try:
        r = requests.get("http://127.0.0.1:54321/quicktunnels").json()
        print("🌐 Cloudflare Tunnel:", r["url"])
    except:
        print("⚠️ URL 조회 실패. 몇 초 더 기다리세요.")

threading.Thread(target=run_cloudflare, daemon=True).start()
threading.Thread(target=find_url, daemon=True).start()


# ======================================================================
# 5. Flask 시작
# ======================================================================
app.run(host="0.0.0.0", port=5000)


모델 로딩 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 모델 로드 완료!
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


⚠️ URL 조회 실패. 몇 초 더 기다리세요.
